# Lesson 3: Classification

Predicting categories with logistic regression.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, auc, roc_auc_score,
                             accuracy_score, precision_score, recall_score, f1_score)

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. The Sigmoid Function

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 100)
plt.figure(figsize=(8, 4))
plt.plot(z, sigmoid(z), 'b-', linewidth=2)
plt.axvline(0, color='gray', linestyle='--')
plt.axhline(0.5, color='gray', linestyle='--')
plt.xlabel('z = β₀ + β₁x₁ + ... + βₚxₚ')
plt.ylabel('σ(z) = P(y=1)')
plt.title('Sigmoid Function')
plt.show()

## 2. Decision Boundary Visualization

In [ ]:
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                            n_clusters_per_class=1, class_sep=1.5, random_state=42)

model = LogisticRegression()
model.fit(X, y)

xx, yy = np.meshgrid(np.linspace(X[:, 0].min()-0.5, X[:, 0].max()+0.5, 100),
                     np.linspace(X[:, 1].min()-0.5, X[:, 1].max()+0.5, 100))
Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, levels=20, cmap='RdBu')
plt.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=2)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', edgecolors='k', alpha=0.7)
plt.colorbar(label='P(y=1)')
plt.title('Decision Boundary')
plt.show()

## 3. Breast Cancer Classification

In [ ]:
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target
print(f"Classes: {data.target_names}")
print(f"Samples: {X_bc.shape[0]}, Features: {X_bc.shape[1]}")
print(f"Class distribution: {np.bincount(y_bc)}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc
)

model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=data.target_names))

## 4. ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.show()

## 5. Threshold Tuning

In [ ]:
thresholds = np.linspace(0, 1, 101)
f1_scores = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    f1_scores.append(f1_score(y_test, y_pred_t))

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_scores)
plt.scatter(best_threshold, f1_scores[best_idx], color='red', s=100,
            label=f'Best threshold = {best_threshold:.2f}, F1 = {f1_scores[best_idx]:.3f}')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.legend()
plt.title('F1 Score vs. Decision Threshold')
plt.show()

## 6. Biotechnology Example: Drug Response

In [ ]:
n = 500
bio = pd.DataFrame({
    'age': np.random.normal(55, 15, n),
    'biomarker_A': np.random.normal(0, 1, n),
    'biomarker_B': np.random.normal(0, 1, n),
    'gene_X': np.random.exponential(1, n),
})
log_odds = -2 + 0.5*bio['biomarker_A'] - 0.3*bio['biomarker_B'] + 1.5*bio['gene_X']
bio['responder'] = (1 / (1 + np.exp(-log_odds)) > 0.5).astype(int)

X_bio, y_bio = bio.drop('responder', axis=1), bio['responder']
X_tr, X_te, y_tr, y_te = train_test_split(X_bio, y_bio, test_size=0.3, random_state=42)
m = LogisticRegression().fit(X_tr, y_tr)
print(classification_report(y_te, m.predict(X_te)))

## 7. Exercises

### Level 1
What is the range of the sigmoid function? Why is this useful for classification?

### Level 2
Train a logistic regression on the breast cancer dataset and compute precision, recall, and F1 for both classes.

### Level 3
If a rare disease has 1% prevalence and a test has 99% accuracy, is it a good test? What metric matters more?

In [ ]:
# Your Level 2 code

## 8. Coding Challenge

Write `find_optimal_threshold(model, X_val, y_val)` that finds the threshold maximizing F1.

In [ ]:
def find_optimal_threshold(model, X_val, y_val):
    y_proba = model.predict_proba(X_val)[:, 1]
    thresholds = np.linspace(0, 1, 101)
    best_f1, best_t = 0, 0
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        f1 = f1_score(y_val, y_pred)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    return best_t, best_f1

t, f1 = find_optimal_threshold(model, X_test, y_test)
print(f"Optimal threshold: {t:.2f}, F1: {f1:.3f}")

## Summary

- Logistic regression predicts probabilities via sigmoid
- Decision boundary at P = 0.5
- Confusion matrix: TP, FP, TN, FN
- Precision, recall, F1 for nuanced evaluation
- ROC AUC measures ranking quality
- Choose threshold based on business costs